# Problema di Scheduling con Algoritmo Genetico
## Massimizzazione del minimo beneficio dei clienti

In questo notebook implementiamo un algoritmo genetico per risolvere un problema di scheduling
con vincoli di **release time**, durate variabili e coefficienti di beneficio/penalità.  
L’obiettivo è **massimizzare il minimo beneficio (utility) dei clienti**.

In [25]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Tuple
import random
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Configurazione per i grafici
plt.rcParams['figure.figsize'] = (12, 8)
sns.set_style("whitegrid")

## Definizione della classe `Task`

Ogni task ha:
- un **release time** `r_j`
- una **durata** `d_j`
- un **beneficio base** `b_j`
- un **coefficiente di penalità** `a_j`

L’utilità di un task è definita come:

$$ u_j = b_j - a_j \cdot c_j $$

dove $c_j$ è il completion time del task.

In [26]:
@dataclass
class Task:
    id: int
    release_time: float
    duration: float
    benefit_coeff: float
    penalty_coeff: float

    def calculate_utility(self, completion_time: float) -> float:
        return self.benefit_coeff - self.penalty_coeff * completion_time

## Classe `SchedulingProblem`

Gestisce la valutazione di una sequenza di task.  
L’obiettivo è **massimizzare il minimo delle utilità**.

In [27]:
class SchedulingProblem:
    def __init__(self, tasks: List[Task]):
        self.tasks = tasks
        self.n_tasks = len(tasks)

    def evaluate_solution(self, sequence: List[int]):
        current_time = 0.0
        start_times = [0.0] * self.n_tasks
        completion_times = [0.0] * self.n_tasks
        utilities = [0.0] * self.n_tasks

        for task_id in sequence:
            task = self.tasks[task_id]
            start_time = max(current_time, task.release_time)
            completion_time = start_time + task.duration
            start_times[task_id] = start_time
            completion_times[task_id] = completion_time
            utilities[task_id] = task.calculate_utility(completion_time)
            current_time = completion_time

        objective_value = min(utilities)
        return objective_value, utilities, start_times, completion_times

## Algoritmo Genetico

L’algoritmo genetico gestisce:
- creazione popolazione iniziale
- valutazione fitness
- selezione (torneo)
- crossover (Order Crossover)
- mutazione (swap)
- elitismo

In [28]:
class GeneticAlgorithm:
    def __init__(self, problem: SchedulingProblem, 
                 population_size: int = 100,
                 mutation_rate: float = 0.1,
                 crossover_rate: float = 0.8,
                 elite_size: int = 20):
        self.problem = problem
        self.population_size = population_size
        self.mutation_rate = mutation_rate
        self.crossover_rate = crossover_rate
        self.elite_size = elite_size
        self.n_tasks = problem.n_tasks
        self.history = {'best_fitness': [],'avg_fitness': [],'worst_fitness': [],'generation': []}
    def create_individual(self): return list(np.random.permutation(self.n_tasks))
    def create_population(self): return [self.create_individual() for _ in range(self.population_size)]
    def fitness(self, individual): return self.problem.evaluate_solution(individual)[0]
    def tournament_selection(self, population, fitnesses, tournament_size=5):
        tournament_indices = random.sample(range(len(population)), tournament_size)
        tournament_fitnesses = [fitnesses[i] for i in tournament_indices]
        winner_idx = tournament_indices[np.argmax(tournament_fitnesses)]
        return population[winner_idx].copy()
    def order_crossover(self, parent1, parent2):
        size = len(parent1)
        start, end = sorted(random.sample(range(size), 2))
        offspring1 = [-1] * size
        offspring1[start:end] = parent1[start:end]
        pointer = end
        for city in parent2[end:] + parent2[:end]:
            if city not in offspring1:
                offspring1[pointer % size] = city
                pointer += 1
        offspring2 = [-1] * size
        offspring2[start:end] = parent2[start:end]
        pointer = end
        for city in parent1[end:] + parent1[:end]:
            if city not in offspring2:
                offspring2[pointer % size] = city
                pointer += 1
        return offspring1, offspring2
    def swap_mutation(self, individual):
        mutated = individual.copy()
        if random.random() < self.mutation_rate:
            i, j = random.sample(range(len(mutated)), 2)
            mutated[i], mutated[j] = mutated[j], mutated[i]
        return mutated
    def evolve(self, generations=500):
        population = self.create_population()
        best_individual, best_fitness = None, float('-inf')
        print("Inizio evoluzione...\nGen\tBest\tAvg\tWorst\n" + "-" * 30)
        for generation in range(generations):
            fitnesses = [self.fitness(ind) for ind in population]
            current_best_idx = np.argmax(fitnesses)
            current_best_fitness = fitnesses[current_best_idx]
            if current_best_fitness > best_fitness:
                best_fitness = current_best_fitness
                best_individual = population[current_best_idx].copy()
            self.history['generation'].append(generation)
            self.history['best_fitness'].append(current_best_fitness)
            self.history['avg_fitness'].append(np.mean(fitnesses))
            self.history['worst_fitness'].append(min(fitnesses))
            if generation % 50 == 0:
                print(f"{generation}\t{current_best_fitness:.3f}\t{np.mean(fitnesses):.3f}\t{min(fitnesses):.3f}")
            new_population = []
            elite_indices = np.argsort(fitnesses)[-self.elite_size:]
            for idx in elite_indices:
                new_population.append(population[idx].copy())
            while len(new_population) < self.population_size:
                if random.random() < self.crossover_rate:
                    parent1 = self.tournament_selection(population, fitnesses)
                    parent2 = self.tournament_selection(population, fitnesses)
                    child1, child2 = self.order_crossover(parent1, parent2)
                    child1 = self.swap_mutation(child1)
                    child2 = self.swap_mutation(child2)
                    new_population.extend([child1, child2])
                else:
                    parent = self.tournament_selection(population, fitnesses)
                    child = self.swap_mutation(parent)
                    new_population.append(child)
            population = new_population[:self.population_size]
        print(f"\nEvoluzione completata!\nMiglior fitness finale: {best_fitness:.6f}")
        return best_individual, best_fitness

## Funzioni di supporto

In [29]:
def generate_random_problem(n_tasks: int = 10, seed: int = None) -> SchedulingProblem:
    if seed:
        np.random.seed(seed)
        random.seed(seed)
    tasks = []
    for i in range(n_tasks):
        task = Task(
            id=i,
            release_time=np.random.exponential(5.0),
            duration=np.random.uniform(1.0, 10.0),
            benefit_coeff=np.random.uniform(20.0, 100.0),
            penalty_coeff=np.random.uniform(0.1, 2.0))
        tasks.append(task)
    return SchedulingProblem(tasks)

## Esempio di utilizzo completo

In [30]:
if __name__ == "__main__":
    problem = generate_random_problem(n_tasks=10, seed=42)
    ga = GeneticAlgorithm(problem, population_size=50, mutation_rate=0.1)
    best_solution, best_fitness = ga.evolve(generations=100)
    print('Best fitness:', best_fitness)
    print('Best solution:', best_solution)

Inizio evoluzione...
Gen	Best	Avg	Worst
------------------------------
0	-6.252	-52.607	-95.036
50	-1.605	-10.652	-86.401

Evoluzione completata!
Miglior fitness finale: -1.605139
Best fitness: -1.6051388460696039
Best solution: [np.int64(1), np.int64(2), np.int64(9), np.int64(7), np.int64(4), np.int64(6), np.int64(5), np.int64(8), np.int64(3), np.int64(0)]
